# Marketing ROI - Simple Linear Regression Analysis
This notebook provides a data-driven approach to evaluating marketing channel performance.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.stats import shapiro

df = pd.read_csv("marketing_and_sales_evaluate_lr.csv")
df = df.dropna()
print("Dataset Shape After Cleaning:", df.shape)

## 1. Exploratory Data Analysis & Visualizations

In [2]:
# Select independent variable based on correlation
correlations = df.corr(numeric_only=True)["Sales"].drop("Sales")
best_predictor = correlations.abs().idxmax()

# 1. Scatter plot for linearity check
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x=best_predictor, y="Sales")
plt.title(f"Sales vs {best_predictor}")
plt.show()

# 2. Histogram for dependent variable distribution
plt.figure(figsize=(6, 4))
sns.histplot(df["Sales"], kde=True)
plt.title("Distribution of Sales")
plt.show()

## 2. Model Building (OLS Regression)

In [3]:
X = df[[best_predictor]]
y = df["Sales"]
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

## 3. Model Interpretation
Based on the OLS model summary:
- **R-squared**: The R-squared value indicates the proportion of variance in Sales that can be explained by the marketing spend. A higher value means a stronger fit.
- **Coefficient**: For every 1-unit increase in marketing budget, Sales change by the coefficient value of the independent variable.

## 4. Regression Assumption Diagnostics

In [4]:
residuals = model.resid

# 3. Q-Q Plot for Normality
fig, ax = plt.subplots(figsize=(6, 4))
sm.qqplot(residuals, line='s', ax=ax)
plt.title("Normal Q-Q Plot")
plt.show()

# 4. Residuals vs Fitted Plot for Homoscedasticity
plt.figure(figsize=(6, 4))
plt.scatter(model.fittedvalues, residuals)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values")
plt.show()

# Formal Statistical Tests
stat, p_value = shapiro(residuals)
print("Shapiro-Wilk Normality P-value:", p_value)
bp_test = het_breuschpagan(residuals, model.model.exog)
print("Breusch-Pagan Homoscedasticity P-value:", bp_test[1])

# Final Linear Equation String Output
print(f"Linear Equation: y = {round(model.params[1], 4)} * x + {round(model.params[0], 4)}")